In [1]:
import pandas as pd

In [2]:
df=pd.read_json('/content/idmanual.json')

In [3]:
df

,id_tx,class_id,description,status
0,009-4140,009,Bank note acceptors for separating good bank n...,A
1,009-4136,009,Fingerprint imagers,A
2,009-4133,009,Laboratory swabs [laboratory instruments],A
3,009-4131,009,Ear plugs for divers,A
4,009-4130,009,DVD recorders,A
...,...,...,...,...
58697,004-664,004,Cutting fluids for milling,A
58698,004-665,004,Cutting oils for millworking,A
58699,004-666,004,Cutting oils for milling,A
58700,004-667,004,Palm oil being biodiesel fuel,A


In [4]:
df.isnull().sum()

,0
id_tx,0
class_id,0
description,0
status,0


In [5]:
status_counts = df['status'].value_counts()
status_counts

,count
status,
A,49477
M,3815
X,2984
D,2426


In [6]:
from sklearn.preprocessing import LabelEncoder
status_encoder = LabelEncoder()
df['status'] = status_encoder.fit_transform(df['status'])

In [7]:
status_counts = df['status'].value_counts()
status_counts

,count
status,
0,49477
2,3815
3,2984
1,2426


In [9]:
label_encoder = LabelEncoder()
df['class_id'] = label_encoder.fit_transform(df['class_id'])

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58702 entries, 0 to 58701
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id_tx        58702 non-null  object
 1   class_id     58702 non-null  int64 
 2   description  58702 non-null  object
 3   status       58702 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 1.8+ MB


In [11]:
df = df.drop(columns=['id_tx'])

In [12]:
df

,class_id,description,status
0,9,Bank note acceptors for separating good bank n...,0
1,9,Fingerprint imagers,0
2,9,Laboratory swabs [laboratory instruments],0
3,9,Ear plugs for divers,0
4,9,DVD recorders,0
...,...,...,...
58697,4,Cutting fluids for milling,0
58698,4,Cutting oils for millworking,0
58699,4,Cutting oils for milling,0
58700,4,Palm oil being biodiesel fuel,0


In [13]:
import re
def clean_text(text):
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    return text

In [14]:
df['description'] = df['description'].apply(clean_text)

In [15]:
df.head()

,class_id,description,status
0,9,bank note acceptors for separating good bank n...,0
1,9,fingerprint imagers,0
2,9,laboratory swabs laboratory instruments,0
3,9,ear plugs for divers,0
4,9,dvd recorders,0


In [61]:
import numpy as np
np.save('label_classes.npy', label_encoder.classes_)

In [18]:
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
import torch
from torch.utils.data import TensorDataset, DataLoader

In [19]:
train_df, val_test_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(val_test_df, test_size=0.5, random_state=42)

In [20]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [23]:
def tokenize_data(df):
    return tokenizer(df['description'].tolist(), padding=True, truncation=True, return_tensors='pt')

In [24]:
train_tokens = tokenize_data(train_df)
val_tokens = tokenize_data(val_df)
test_tokens = tokenize_data(test_df)

In [25]:
train_labels = torch.tensor(train_df['class_id'].values)
val_labels = torch.tensor(val_df['class_id'].values)
test_labels = torch.tensor(test_df['class_id'].values)

In [26]:
train_dataset = TensorDataset(train_tokens['input_ids'], train_labels)
val_dataset = TensorDataset(val_tokens['input_ids'], val_labels)
test_dataset = TensorDataset(test_tokens['input_ids'], test_labels)

In [27]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [30]:
len(df['class_id'].unique())

49

In [28]:
from transformers import BertForSequenceClassification, AdamW
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(df['class_id'].unique()))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [31]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [32]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm

In [33]:
optimizer = AdamW(model.parameters(), lr=2e-5)
loss_fn = CrossEntropyLoss()

In [34]:
epochs = 3
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader):
        optimizer.zero_grad()

        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {avg_train_loss:.4f}")

    # Validation step (optional)
    model.eval()
    total_val_loss = 0
    correct_predictions = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids, labels = batch
            input_ids, labels = input_ids.to(device), labels.to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs.loss
            logits = outputs.logits

            total_val_loss += loss.item()

            # Calculate accuracy
            preds = torch.argmax(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)

    avg_val_loss = total_val_loss / len(val_loader)
    val_accuracy = correct_predictions.double() / len(val_loader.dataset)
    print(f"Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

100%|██████████| 1468/1468 [22:01<00:00,  1.11it/s]


Epoch 1/3, Training Loss: 1.9384
Validation Loss: 0.7456, Validation Accuracy: 0.8082


100%|██████████| 1468/1468 [22:04<00:00,  1.11it/s]


Epoch 2/3, Training Loss: 0.5515
Validation Loss: 0.5055, Validation Accuracy: 0.8654


100%|██████████| 1468/1468 [22:03<00:00,  1.11it/s]


Epoch 3/3, Training Loss: 0.3120
Validation Loss: 0.4340, Validation Accuracy: 0.8872


In [35]:
model.eval()
total_test_loss = 0
correct_predictions = 0
with torch.no_grad():
    for batch in test_loader:
        input_ids, labels = batch
        input_ids, labels = input_ids.to(device), labels.to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        total_test_loss += loss.item()

        # Calculate accuracy
        preds = torch.argmax(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

avg_test_loss = total_test_loss / len(test_loader)
test_accuracy = correct_predictions.double() / len(test_loader.dataset)
print(f"Test Loss: {avg_test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

Test Loss: 0.4330, Test Accuracy: 0.8862


In [36]:
!pip install wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 7.1 MB/s eta 0:00:00


In [37]:
import wandb
wandb.init(project="trademark-classification", entity="sriharish-r2020-vellore-institute-of-technology")

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [38]:
# Save model checkpoint
torch.save(model.state_dict(), "model.pth")

# Log the model
wandb.save("model.pth")

['/content/wandb/run-20240814_053058-551vd8s6/files/model.pth']

In [62]:

user_input = input("Enter text to classify: ")


user_input_cleaned = clean_text(user_input)
user_input_tokenized = tokenizer(user_input_cleaned, padding='max_length', truncation=True, return_tensors='pt')

# Move input to device
user_input_tokenized = {key: val.to(device) for key, val in user_input_tokenized.items()}

# Make prediction
model.eval()
with torch.no_grad():
    outputs = model(**user_input_tokenized)
    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

# Decode prediction
predicted_class = label_encoder.inverse_transform([predicted_class_id])[0]
print(f"Predicted Class: {predicted_class}")


Enter text to classify: corn flakes
Predicted Class: 030
